In [1]:
import numpy as np
from astropy import units as u
from astropy.coordinates import SkyCoord
from astropy.time import Time
from astroquery.jplhorizons import Horizons
from astroquery.mpc import MPC
from scipy.interpolate import interp1d
from sora import Body, Observer
from sora.ephem.meta import BaseEphem
from sora.prediction import prediction

SORA version: 0.3.3


In [2]:
# Hiperparámetros
rock = 'Silesia'
obs = 'W63'
epoch_jpl = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1m'
}
epoch_mpc = {
  'start': '2019-06-27',
  'stop': '2019-06-29',
  'step': '1min'
}
mag_lim = 16
sora_step = 10 # Paso en segundos del muestreo de predicción de SORA

In [3]:
# Bajamos efemérides del asteroide con horizons
body_jpl = Horizons(id=rock, epochs=epoch_jpl, location=obs)
eph_jpl = body_jpl.ephemerides()
data_jpl = eph_jpl['datetime_jd', 'RA', 'DEC', 'delta'] # Datos que pide SORA en objeto efemérides
sigma_jpl = eph_jpl['RA_3sigma', 'DEC_3sigma'] # incertidumbre 3sigma para la ascención recta y declinación en ese instante
error_jpl = eph_jpl['SMAA_3sigma', 'SMIA_3sigma', 'Theta_3sigma'] # característica del elipse de error (matriz de covarianza diagonalizada)

# Bajamos efemérides del asteroide con MPCES
body_mpc = MPC.query_object('asteroid', name=rock)
eph_mpc = MPC.get_ephemeris(
  rock, 
  step=epoch_mpc['step'], 
  start=epoch_mpc['start'], 
  number=1441, 
  location=obs
)
eph_mpc['Date_jd'] = Time(eph_mpc['Date']).jd # Las fechas tienen que estar en formato juliano
data_mpc = eph_mpc['Date_jd', 'RA', 'Dec', 'Delta'] #  Datos que pide SORA en objeto efemérides
error_mpc = eph_mpc['Uncertainty 3sig', 'Unc. P.A.'] # características del error principal (componente principal)

In [4]:
# EphemTable class
class EphemTable(BaseEphem):
  def __init__(self, table, name=None, spkid=None, radius=None, error_ra=0, error_dec=0, H=None, G=None, **kwargs):
    
    # Handle kwargs (compatibility with base clase BaseEphem)
    base_kwargs = kwargs.copy()
    if radius is not None:
      base_kwargs['radius'] = radius
    if H is not None:
      base_kwargs['H'] = H
    if G is not None:
      base_kwargs['G'] = G
    base_kwargs['error_ra'] = error_ra
    base_kwargs['error_dec'] = error_dec
    super().__init__(name=name, spkid=spkid, **base_kwargs)
    
    # Check if the table is empty
    if len(table) == 0:
      raise ValueError("No data found.")
    
    # Data columns necessary
    cols = ['time', 'ra', 'dec', 'distance']
    
    # Validate table columns
    for col in cols:
      
      # They must exist
      if col not in table.columns:
        raise ValueError(f"'{col}' not found in data.")
      
      # They must not contain nans or infinite values
      if not np.all(np.isfinite(table[col])):
        raise ValueError(f"'{col}' contains NaN or infinite values.")
      
    # Time must not be negative
    if table['time'].min() < 0:
      raise ValueError("Time cannot be negative")
      
    # Validate ra being between 0 and 360 degrees
    if table['ra'].min() < 0:
      raise ValueError("RA coordinates must not be negative.")
    if table['ra'].max() > 360:
      raise ValueError("RA coordinates cannot exceed 360°.")
    
    # Validate dec being between -90 and 90 degrees
    if table['dec'].min() < -90:
      raise ValueError("DEC coordinates cannot be lower than -90°.")
    if table['dec'].max() > 90:
      raise ValueError("DEC coordinates cannot exceed 90°.")
    
    # Distance must be positive
    if table['distance'].min() <= 0:
      raise ValueError("Distance cannot be negative or 0.")
    
    # The time must be in increasing order
    if not np.all(np.diff(table['time']) > 0):
      raise ValueError("Time values must be in strictly increasing order.")
  
    # Validate units
    if not table['time'].unit or table['time'].unit != u.d:
      raise ValueError("Time must be given in Julian Date (JD).")
    if not table['ra'].unit or table['ra'].unit != u.deg:
      raise ValueError("RA coordinates must be in degrees.")
    if not table['dec'].unit or table['dec'].unit != u.deg:
      raise ValueError("DEC coordinates must be in degrees.")
    if not table['distance'].unit or table['distance'].unit != u.au:
      raise ValueError("Distance must be in AU.")
    
    # Store sine and cosine of RA for circular interpolation
    table['ra_sin'] = np.sin(np.deg2rad(table['ra']))
    table['ra_cos'] = np.cos(np.deg2rad(table['ra']))
      
    # Save arg data
    self.table = table
    
    # Save starting date and ending date
    self.min_time = table['time'].min()
    self.max_time = table['time'].max()
    
    # Create meta attribute
    self.meta = {'kernels': 'EphemTable'}
    
    # Convert times to numeric values
    times = list(table['time'])
    
    # Create linear interpolators for each variable (extrapolation is not supported)
    self._inter_ra_sin = interp1d(times, table['ra_sin'], bounds_error=True)
    self._inter_ra_cos = interp1d(times, table['ra_cos'], bounds_error=True)
    self._inter_dec = interp1d(times, table['dec'], bounds_error=True)
    self._inter_distance = interp1d(times, table['distance'], bounds_error=True)
    
  # Method to compute position of an object for a given time
  def get_position(self, time, observer='geocenter'):
    
    # The given time must be a Time object
    if not isinstance(time, Time):
      raise TypeError("Time must be an astropy Time object.")
    
    # Convert time to juliand dates
    jd = time.jd 
    
    # Compute corresponding coordinate and distance for the given time using the linear interpolator
    ra_sin = self._inter_ra_sin(jd)
    ra_cos = self._inter_ra_cos(jd)
    dec = self._inter_dec(jd)
    distance = self._inter_distance(jd)
    
    # Reconstruct RA coordinate from interpolated sine and cosine
    ra = np.rad2deg(np.arctan2(ra_sin, ra_cos)) % 360
    
    # Return computed values as a skycoord
    return SkyCoord(
      ra=ra * u.deg,
      dec=dec * u.deg,
      distance=distance * u.au,
      frame="icrs"
    )

In [20]:
# Renombramos columnas para que se pueda instanciar EphemTable
# data_jpl.rename_columns(["datetime_jd", "RA", "DEC", "delta"], ["time", "ra", "dec", "distance"])
# data_mpc.rename_columns(["Date_jd", "RA", "Dec", "Delta"], ["time", "ra", "dec", "distance"])

# La conversión de MPC no le dio unidades oops
# data_mpc['time'].unit = u.d

# Instanciamos EphemTable para cada caso
eph_jpl_sora = EphemTable(data_jpl)
eph_mpc_sora = EphemTable(data_mpc)

# Variable controlada (objeto)
sora_body = Body(rock)

# Instanciamos objeto menor con SORA utilizando los datos de jpl y mpc
sora_body_jpl = Body(rock, ephem=eph_jpl_sora)
sora_body_mpc = Body(rock, ephem=eph_mpc_sora)

# Instanciamos observador
sora_obs = Observer(name=obs, code=obs)

Obtaining data for Silesia from SBDB
Obtaining data for Silesia from SBDB


ValueError: Cannot set "ephem" with <class '__main__.EphemTable'>. Allowed types are: [<class 'sora.ephem.core.EphemPlanete'>, <class 'sora.ephem.core.EphemKernel'>, <class 'sora.ephem.core.EphemJPL'>, <class 'sora.ephem.core.EphemHorizons'>]

In [ ]:
# Controlado
sora_pred = prediction(
  body=sora_body,
  reference_center=sora_obs,
  time_beg=Time(epoch_jpl['start']),
  time_end=Time(epoch_jpl['stop']),
  mag_lim=mag_lim,
  divs=3,
  radius=300,
  step=10,
  verbose=True
)

Ephemeris was split in 3 parts for better search of stars

Searching occultations in part 1/3
Generating Ephemeris between 2019-06-27 00:00:00.000 and 2019-06-27 15:59:50.000 ...
    46 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 2/3
Generating Ephemeris between 2019-06-27 16:00:00.000 and 2019-06-28 07:59:50.000 ...
    60 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 3/3
Generating Ephemeris between 2019-06-28 08:00:00.000 and 2019-06-28 23:59:50.000 ...
    53 GaiaDR3 stars downloaded
Identifying occultations ...

1 occultations found.


In [ ]:
# Experimento
jpl_pred = prediction(
  body=sora_body_jpl,
  reference_center=sora_obs,
  time_beg=Time(epoch_jpl['start']),
  time_end=Time(epoch_jpl['stop']),
  mag_lim=mag_lim,
  divs=3,
  radius=300,
  step=10,
  verbose=True
)

Ephemeris was split in 3 parts for better search of stars

Searching occultations in part 1/3
Generating Ephemeris between 2019-06-27 00:00:00.000 and 2019-06-27 15:59:50.000 ...
    46 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 2/3
Generating Ephemeris between 2019-06-27 16:00:00.000 and 2019-06-28 07:59:50.000 ...
    60 GaiaDR3 stars downloaded
Identifying occultations ...

Searching occultations in part 3/3
Generating Ephemeris between 2019-06-28 08:00:00.000 and 2019-06-28 23:59:50.000 ...
    53 GaiaDR3 stars downloaded
Identifying occultations ...

No stellar occultation was found.


In [22]:
import inspect

from sora.body.core import Body

print(inspect.getsource(Body))

class Body(BaseBody):
    """Represent and manage information for a Solar System body.

    Parameters
    ----------
    name : `str`, required
        Name of the object. It can also be the `spkid` or designation number
        used to query the SBDB (Small-Body DataBase). In this case, the name is
        case-insensitive.

    database : `str`, `None`, optional, default='auto'
        Database used to query the object. It can be ``'satdb'`` for the
        temporary hardcoded satellite database, ``'sbdb'`` to query the SBDB, or
        ``'auto'`` to try ``'satdb'`` first and then ``'sbdb'``. If the user
        wants to provide the object information locally, ``database`` must be
        ``None`` and `spkid` must be given.

    ephem : `sora.EphemKernel`, `sora.EphemHorizons`, `sora.EphemJPL`, `sora.EphemPlanete`
        Ephemeris object that contains information about the object's
        ephemeris. It can also be ``'horizons'`` to automatically define an
        `sora.EphemHorizo

In [23]:
import inspect

from sora.body.core import Body

source = inspect.getsource(Body)

for i, line in enumerate(source.splitlines(), 1):
    if "ephem" in line.lower():
        print(f"{i}: {line}")

18:     ephem : `sora.EphemKernel`, `sora.EphemHorizons`, `sora.EphemJPL`, `sora.EphemPlanete`
19:         Ephemeris object that contains information about the object's
20:         ephemeris. It can also be ``'horizons'`` to automatically define an
21:         `sora.EphemHorizons` object, or a list of kernels to automatically
22:         define an `sora.EphemKernel` object.
32:         If ``database=None``, the user must give a `spkid` or an `ephem` object
94:                           "orbit_class", "spkid", "tholen", "ephem", "frame", "shape"]
96:         self._shared_with = {'ephem': {}, 'occultation': {}}
128:         self._shared_with['ephem']['search_name'] = self._search_name
129:         self._shared_with['ephem']['id_type'] = self._id_type
139:         if 'ephem' not in kwargs:
140:             self.ephem = 'horizons'
290:         return self.ephem.get_position(time=time, observer=observer)
315:         obj = self.ephem.get_position(time, observer=observer)
359:             ep